# 对比实验：裸写 SDK vs Microsoft Agent Framework

**目的**：亲手体会「Agent Framework 到底替你封装了什么」。

**任务**：问 Agent「东京能预订吗？如果可以，再告诉我温哥华呢？」
- **Part A**：原生 `openai` SDK，手写 Agent Loop
- **Part B**：Microsoft Agent Framework 重写
- 最后看对比表，你就知道框架省了哪些活

> 这一节不是「学会 MAF」，而是「理解 Framework 存在的意义」。Foundry Agent Service（微软托管运行平台）这一层我们用 Kimi API 替代了，所以只关注 MAF 这层 SDK。

## Part A — 原生 openai SDK（无框架）

下面**所有事都要你自己写**：
1. 建立与 Kimi 的连接
2. 手写工具的 JSON Schema
3. 手写 Agent Loop（调 LLM → 解析 tool_calls → 执行 → 回灌 → 再调）
4. 自己维护 `messages` 上下文

逐行看注释里标了「框架会替你做的事」。

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)
MODEL = os.environ["LLM_MODEL"]


def check_destination_availability(destination: str) -> str:
    """检查某个度假目的地当前是否可预订。"""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} {'可预订' if is_available else '不可预订'}。"


tools = [
    {
        "type": "function",
        "function": {
            "name": "check_destination_availability",
            "description": "检查某个度假目的地当前是否可预订。",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination": {
                        "type": "string",
                        "description": "要查询可用性的目的地",
                    }
                },
                "required": ["destination"],
            },
        },
    }
]

# 显式声明 messages 的类型为 list[dict]，避免类型推导过于收窄
messages: list[dict] = [
    {
        "role": "system",
        "content": "你是一个旅行预订代理。在推荐前务必检查目的地可用性。",
    },
    {
        "role": "user",
        "content": "东京能预订吗？如果可以，再告诉我温哥华呢？",
    },
]

print("=== 开始原生 SDK 手写 Agent Loop ===")

for turn in range(5):
    # 用 # type: ignore 规避消息列表类型告警
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, temperature=1.0  # type: ignore
    )
    msg = resp.choices[0].message

    if msg.tool_calls:
        # 1. 使用 model_dump() 转成标准 dict 追加，彻底解决 ChatCompletionMessage 类型冲突
        messages.append(msg.model_dump())

        for tc in msg.tool_calls:
            # 2. 使用 getattr 安全获取 function 属性，彻底解决 CustomToolCall 属性未知问题
            func = getattr(tc, "function", None)
            if func:
                args = json.loads(func.arguments)
                print(f"[工具调用] {func.name}({args})")
                result = check_destination_availability(**args)
                print(f"[工具结果] {result}")

                # 3. 追加 tool 消息
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id, "content": result}
                )
    else:
        print(f"智能体：{msg.content}")
        break

### Part A 你手写了多少？
- 连接客户端：自己写
- 工具描述：手写 JSON Schema（冗长、易错）
- 循环：自己写 `for` + 判断 `tool_calls`
- 上下文：自己 `append` messages
- 错误/重试/日志：这里都没写（真实项目还要补一大堆）

## Part B — Microsoft Agent Framework

同样任务，看框架帮你省了什么。代码量一下子就下来了。

In [ ]:
import os, asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
load_dotenv()

# ① 连接模型（provider 帮你封装了 client 细节）
provider = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

# ② @tool 装饰器：类型注解 + 文档字符串自动变成工具描述（不用手写 JSON Schema）
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "要查询可用性的目的地"]
) -> str:
    """检查某个度假目的地当前是否可预订。"""
    available = {"Barcelona": True, "Tokyo": True, "Cape Town": False,
                 "Vancouver": True, "Dubai": False}
    is_available = available.get(destination, False)
    return f"{destination} {'可预订' if is_available else '不可预订'}。"

# ③ 把 client + 人设 + 工具 组合成 agent
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions="你是一个旅行预订代理。在推荐前务必检查目的地可用性。",
    tools=[check_destination_availability],
)

async def main():
    # ④ ⑤ session 自动维护上下文；agent.run() 一行搞定 Agent Loop
    session = agent.create_session()
    response = await agent.run(
        "东京能预订吗？如果可以，再告诉我温哥华呢？",
        session=session,
    )
    print(f"智能体：{response}")

asyncio.run(main())


## 对比总结

| 职责 | 原生 SDK（Part A） | MAF（Part B） |
|---|---|---|
| 连接模型 | 自己 new client | provider 封装 |
| 工具定义 | 手写 JSON Schema | `@tool` + 类型注解 |
| Agent Loop | 自己写 for + tool_calls 解析 | `agent.run()` 一行 |
| 上下文 | 自己 append messages | `session` 自动 |
| 多轮 | 自己管理 | `session` 自动 |
| 异步 | 自己管 | 框架管 |
| 代码行数 | ~45 行 | ~20 行 |

**结论**：Framework 不是 Agent 本身，而是把「连接 / 工具 / 循环 / 上下文 / 多轮」这些基础设施标准化。
你学的是**抽象**，不是某一个框架——以后换 LangChain / AutoGen / OpenAI Agents SDK，底层思想都一样。